## Import Libraries

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
from warnings import filterwarnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
filterwarnings("ignore")

## Load Datasets

In [ ]:
train_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

In [ ]:
train_df.info()
test_df.info()

In [ ]:
print("Dataset of train:")
display(train_df)
print("Dataset of test:")
display(test_df)

In [ ]:
missing = train_df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

In [ ]:
# LotFrontage – median
train_df=train_df.drop(columns=["PoolQC","MiscFeature","Alley","Fence","MasVnrType","FireplaceQu"])
test_df=test_df.drop(columns=["PoolQC","MiscFeature","Alley","Fence","MasVnrType","FireplaceQu"])


# Numeric - NA means no feature → 0
num_zero = ["MasVnrArea", "GarageYrBlt",
            "BsmtFullBath", "BsmtHalfBath","LotFrontage",
            "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
            "GarageCars", "GarageArea"]
for col in num_zero:
    train_df[col] = train_df[col].fillna(0)
    test_df[col] = test_df[col].fillna(0)

# Categorical - NA means "None"
cat_none = [
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtFinType2", "BsmtExposure", "BsmtFinType1", "BsmtCond", "BsmtQual", "Electrical",
    "MSZoning", "Utilities", "Functional", "Exterior1st", "Exterior2nd", "KitchenQual", "SaleType"
]

for col in cat_none:
    train_df[col] = train_df[col].fillna("None")
    test_df[col] = test_df[col].fillna("None")

In [ ]:
train_df_plot=train_df.drop('Id', axis=1)
print(train_df_plot['SalePrice'].describe())
plt.figure(figsize=(9, 8))
sns.distplot(train_df_plot['SalePrice'], color='r', bins=100, hist_kws={'alpha': 0.4});

In [ ]:
plt.subplot(1,2,2)
sns.histplot(np.log1p(train_df["SalePrice"]), kde=True)
plt.title("SalePrice — log1p")

plt.tight_layout()
plt.show()

## Separate target and features

In [ ]:
y_train = train_df['SalePrice']
x_train = train_df.drop(['SalePrice'], axis=1)
x_test = test_df.copy()

## Preprocessing

In [ ]:
# Combine for preprocessing
all_data = pd.concat([x_train, x_test], axis=0)
all_data=all_data.drop(columns=["Id"])


#Handle missing values
for col in all_data.columns:
    if all_data[col].dtype == 'object':  # categorical
        all_data[col] = all_data[col].fillna('None')
    else:  # numeric
        all_data[col] = all_data[col].fillna(all_data[col].median())

# One-hot encode categorical variables
all_data = pd.get_dummies(all_data)

# Split back into train and test
x_train = all_data.iloc[:len(train_df), :]
x_test = all_data.iloc[len(train_df):, :]

## 1. Random Forest 

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1,
)

In [ ]:
rf_model=rf.fit(x_train,y_train)
# Test: 80 columns
y_test = rf_model.predict(x_test)

# Add predicts to dataset of test 
test1_df= test_df.copy()
test1_df["SalePrice"] = y_test

In [ ]:
test1_df.head()

## Submission

In [ ]:
test1_df=test1_df[["Id","SalePrice"]]

In [ ]:
test1_df.head()

In [ ]:
test1_df.to_csv("submission.csv", index=False) 